<a href="https://colab.research.google.com/github/Shibainulove/6070-001-202630/blob/main/WeeklyModules/Week06/Student_Performance_Lambda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A lambda function for the Student Performance Data Set

A lambda function that takes the student performance data frame and returns a new data frame comparing the average scores of students who did and didn't complete the test preparation course.

In [1]:
# Colab setup (Git_Key method). Opening this notebook from GitHub in Colab brings in only the
# .ipynb file, not the data files next to it. In Colab, this cell clones the repository with the
# GitHub token saved in Colab Secrets as Git_Key, then moves into this notebook's folder so the
# relative file paths below work as they do locally. Run locally, it changes nothing.
import os
import subprocess
from pathlib import Path

GITHUB_USER = 'Shibainulove'
REPO = '6070-001-202630'
NOTEBOOK_DIR = 'WeeklyModules/Week06'  # this notebook's folder in the repository

try:
    from google.colab import userdata
except ImportError:
    userdata = None  # not running in Colab

if userdata is None:
    print('Running locally: using the files next to this notebook')
else:
    repo_dir = Path('/content') / REPO
    if not repo_dir.exists():
        try:
            token = userdata.get('Git_Key')
        except Exception:  # no Git_Key secret, or this notebook was not given access to it
            token = ''  # cloning without a token still works while the repository is public
        login = f'{token}@' if token else ''
        url = f'https://{login}github.com/{GITHUB_USER}/{REPO}.git'
        clone = subprocess.run(['git', 'clone', '--depth', '1', url, str(repo_dir)],
                               capture_output=True, text=True,
                               env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
        if clone.returncode != 0:
            details = clone.stderr.replace(token, '***') if token else clone.stderr
            raise RuntimeError('Could not clone the repository. In Colab Secrets (key icon), save a '
                               'GitHub token that can read it as Git_Key and turn on notebook access.\n'
                               + details)
        # Remove the token from the clone's saved remote address
        subprocess.run(['git', '-C', str(repo_dir), 'remote', 'set-url', 'origin',
                        f'https://github.com/{GITHUB_USER}/{REPO}.git'], check=True)
    os.chdir(repo_dir / NOTEBOOK_DIR)
    print('Reading files from', Path.cwd())

Reading files from /content/6070-001-202630/WeeklyModules/Week06


In [2]:
import pandas as pd

# Student_Performance_Data.csv is in the same folder as this notebook. In Colab, the setup
# cell above clones the repository and moves into this folder first.
students = pd.read_csv('Student_Performance_Data.csv')
print(students.shape)
students.head()

(1000, 10)


,gender,race/ethnicity,parental level of education,lunch,math score,reading score,writing score,total score,test preparation course,admission prospect
0,female,group B,bachelor's degree,standard,72,72,74,218,none,medium
1,female,group C,some college,standard,69,90,88,247,completed,high
2,female,group B,master's degree,standard,90,95,93,278,none,high
3,male,group A,associate's degree,free/reduced,47,57,44,148,none,low
4,male,group C,some college,standard,76,78,75,229,none,high


## The lambda function

In [3]:
# Takes the student performance data frame (df) and returns a new data frame with the
# average math, reading, writing and total scores for each test preparation group
scores_by_prep = lambda df: (
    df.groupby('test preparation course')[['math score', 'reading score', 'writing score', 'total score']]
    .mean()
    .round(1)
)

prep_summary = scores_by_prep(students)
prep_summary

,math score,reading score,writing score,total score
test preparation course,,,,
completed,69.7,73.9,74.4,218.0
none,64.1,66.5,64.5,195.1


In [4]:
# The result is a separate data frame; the original data is unchanged
print(type(prep_summary))
print('Original data:', students.shape, '| New data frame:', prep_summary.shape)

<class 'pandas.core.frame.DataFrame'>
Original data: (1000, 10) | New data frame: (2, 4)


**What the new data frame shows:** students who completed the test preparation course scored higher on all three tests, by about 5.6 points in math, 7.4 in reading and 9.9 in writing, or 22.9 points in total. This is an observational comparison, so it doesn't prove that the course caused the difference: students who chose to take it may differ in other ways.